# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deeepanshchandra/flyrank-aiml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule

I will prioritize pages that are both visible and show a clear refresh opportunity. A page gets a higher baseline score when it has enough search impressions, is stale based on time since its last update, and has low CTR for its search position.

The rule is intentionally simple and uses no fitted weights or future outcome information. The main reason code is `stale_visible_low_ctr`, meaning the page is old, still receives meaningful search visibility, and has a low CTR relative to its position. The action is `refresh_and_review_ctr`.

Pages that do not meet the full condition receive a lower score and are labeled `monitor`.

In [9]:
import os
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

# Make the repository available in Colab.
REPO_DIR = "flyrank-aiml-internship"
REPO_URL = "https://github.com/deeepanshchandra/flyrank-aiml-internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Label is used only for evaluation, never as a scoring input.
df["is_declining_label"] = (
    df["trend_direction"].str.lower().eq("down").astype(int)
)

print(f"Pages loaded: {len(df):,}")
print(f"Declining base rate: {df['is_declining_label'].mean():.3f}")

print("\nSignals used by the rule:")
print([
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
])

Pages loaded: 30,000
Declining base rate: 0.542

Signals used by the rule:
['days_since_last_update', 'impressions_90d', 'avg_position', 'ctr']


In [10]:
# Signal check 1: staleness
# Verdict will be based on whether older pages show more decline.

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 89, 179, np.inf],
    labels=["<90 days", "90-179 days", "180+ days"]
)

staleness_check = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("is_declining_label", "size"),
          declining_rate=("is_declining_label", "mean"),
          median_impressions=("impressions_90d", "median")
      )
      .reset_index()
)

print("Signal 1: staleness")
display(staleness_check)

Signal 1: staleness


,staleness_bucket,n,declining_rate,median_impressions
0,<90 days,20655,0.512031,472.0
1,90-179 days,9171,0.611057,1692.0
2,180+ days,174,0.471264,15.5


### Signal 1 verdict: MIXED

Staleness shows a mixed relationship with decline. The declining rate rises from 51.2% for pages updated within 90 days to 61.1% for pages updated 90–179 days ago. However, the 180+ day bucket has a lower declining rate of 47.1% and contains only 174 pages. Therefore, staleness is useful as a directional signal, but it is not strong enough to treat as a simple monotonic rule by itself.

In [11]:
# Signal check 2: CTR relative to average position.

df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[-np.inf, 10, 20, 30, np.inf],
    labels=["Top 10", "11-20", "21-30", "31+"]
)

# Use fixed CTR buckets so repeated CTR values do not break qcut.
df["ctr_bucket"] = pd.cut(
    df["ctr"],
    bins=[-np.inf, 0.02, 0.05, 0.10, np.inf],
    labels=["Very low CTR", "Low CTR", "Medium CTR", "High CTR"]
)

ctr_position_check = (
    df.groupby(["position_bucket", "ctr_bucket"], observed=False)
      .agg(
          n=("is_declining_label", "size"),
          declining_rate=("is_declining_label", "mean")
      )
      .reset_index()
)

print("Signal 2: CTR relative to average position")
display(ctr_position_check)

Signal 2: CTR relative to average position


,position_bucket,ctr_bucket,n,declining_rate
0,Top 10,Very low CTR,5899,0.446008
1,Top 10,Low CTR,290,0.706897
2,Top 10,Medium CTR,775,0.665806
3,Top 10,High CTR,7224,0.549142
4,11-20,Very low CTR,2884,0.640083
5,11-20,Low CTR,209,0.669856
6,11-20,Medium CTR,545,0.656881
7,11-20,High CTR,3635,0.574691
8,21-30,Very low CTR,1642,0.582217
9,21-30,Low CTR,199,0.768844


### Signal 2 verdict: MIXED

CTR relative to average position shows a mixed relationship with decline. The declining rate varies substantially across CTR buckets within the same position ranges, but the direction is not consistent across all position ranges. For example, in the Top 10, very-low CTR has a 44.6% declining rate while low and medium CTR are higher, whereas in positions 31+ the high-CTR group has a lower declining rate of 45.8% compared with 71.9% for low CTR. Therefore, CTR should be considered together with position rather than used as a simple standalone threshold.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
# Build ONE simple baseline rule using February signals only.

import numpy as np
from pathlib import Path

baseline = df.copy()

# Score pages using three observable February signals.
# Higher score = higher review priority.

baseline["score"] = (
    40 * baseline["days_since_last_update"].fillna(0).clip(0, 365) / 365
    + 30 * (1 - baseline["ctr"].fillna(baseline["ctr"].median()).clip(0, 1))
    + 30 * (1 - baseline["avg_position"].fillna(baseline["avg_position"].median()).clip(1, 100) / 100)
)

# One reason code only.
baseline["reason_code"] = np.select(
    [
        baseline["days_since_last_update"].fillna(0) >= 90,
        baseline["ctr"].fillna(baseline["ctr"].median()) < 0.05,
        baseline["avg_position"].fillna(baseline["avg_position"].median()) > 20
    ],
    [
        "stale_content",
        "low_ctr",
        "weak_visibility"
    ],
    default="mixed_signal"
)

# One action label.
baseline["action"] = np.where(
    baseline["score"] >= baseline["score"].quantile(0.90),
    "refresh_review",
    "monitor"
)

# Rank all pages.
baseline = baseline.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

baseline["rank"] = np.arange(1, len(baseline) + 1)

# Keep the queue useful and compact.
queue_cols = [
    "rank",
    "content_id",
    "score",
    "reason_code",
    "action"
]

queue = baseline[queue_cols].copy()

# Write the required output file.
output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

queue.to_csv(output_path, index=False)

print(f"Rows ranked: {len(queue):,}")
print(f"Output written: {output_path}")
print("\nTop 10:")
display(queue.head(10))

Rows ranked: 30,000
Output written: work/outputs/baseline_action_score.csv

Top 10:


,rank,content_id,score,reason_code,action
0,1,content_1b4ec72dafd4,97.900000,stale_content,refresh_review
1,2,content_55a5b1c46474,97.750000,stale_content,refresh_review
2,3,content_06e19c6486b0,95.102740,stale_content,refresh_review
3,4,content_e2b702f4f92b,93.812740,stale_content,refresh_review
4,5,content_ccf25ed65a99,93.124658,stale_content,refresh_review
5,6,content_02b0d6e30129,92.231370,stale_content,refresh_review
6,7,content_7736e6144f3b,91.755068,stale_content,refresh_review
7,8,content_07ce98c6085a,91.725068,stale_content,refresh_review
8,9,content_f488400fca67,91.714658,stale_content,refresh_review
9,10,content_8f2c815af658,91.066301,stale_content,refresh_review


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [13]:
# Review the top 20 ranked pages.
# The review uses only the baseline signals and makes the uncertainty explicit.

top20 = baseline.head(20).copy()

def confidence_note(row):
    if row["score"] >= baseline["score"].quantile(0.95):
        return "high_score"
    elif row["score"] >= baseline["score"].quantile(0.90):
        return "moderate_score"
    else:
        return "lower_score"

def what_would_make_it_wrong(row):
    if row["reason_code"] == "stale_content":
        return "Recent meaningful updates may not be reflected in the staleness field."
    elif row["reason_code"] == "low_ctr":
        return "Low CTR may be explained by query mix or position rather than a content problem."
    elif row["reason_code"] == "weak_visibility":
        return "Poor position may reflect search demand or competition rather than refresh need."
    else:
        return "The combined signals may not indicate a useful intervention after manual review."

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(
    what_would_make_it_wrong,
    axis=1
)

review_cols = [
    "rank",
    "content_id",
    "score",
    "action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

top20_review = top20[review_cols].copy()

print("Top-20 review:")
display(top20_review)

Top-20 review:


,rank,content_id,score,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_1b4ec72dafd4,97.900000,refresh_review,stale_content,high_score,Recent meaningful updates may not be reflected...
1,2,content_55a5b1c46474,97.750000,refresh_review,stale_content,high_score,Recent meaningful updates may not be reflected...
2,3,content_06e19c6486b0,95.102740,refresh_review,stale_content,high_score,Recent meaningful updates may not be reflected...
3,4,content_e2b702f4f92b,93.812740,refresh_review,stale_content,high_score,Recent meaningful updates may not be reflected...
4,5,content_ccf25ed65a99,93.124658,refresh_review,stale_content,high_score,Recent meaningful updates may not be reflected...
5,6,content_02b0d6e30129,92.231370,refresh_review,stale_content,high_score,Recent meaningful updates may not be reflected...
6,7,content_7736e6144f3b,91.755068,refresh_review,stale_content,high_score,Recent meaningful updates may not be reflected...
7,8,content_07ce98c6085a,91.725068,refresh_review,stale_content,high_score,Recent meaningful updates may not be reflected...
8,9,content_f488400fca67,91.714658,refresh_review,stale_content,high_score,Recent meaningful updates may not be reflected...
9,10,content_8f2c815af658,91.066301,refresh_review,stale_content,high_score,Recent meaningful updates may not be reflected...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [14]:
# Inspect weaker recommendations near the review threshold.
# These rows help test where the simple baseline may make questionable calls.

review_threshold = baseline["score"].quantile(0.90)

weak_picks = (
    baseline[
        (baseline["score"] >= review_threshold * 0.95) &
        (baseline["score"] < review_threshold)
    ]
    .sort_values("score", ascending=False)
    .head(10)
    .copy()
)

weak_picks["why_it_may_be_weak"] = np.select(
    [
        weak_picks["reason_code"] == "stale_content",
        weak_picks["reason_code"] == "low_ctr",
        weak_picks["reason_code"] == "weak_visibility"
    ],
    [
        "Staleness alone may not mean the page needs a refresh.",
        "Low CTR may be explained by search position or query mix.",
        "Weak visibility may reflect competition or low demand rather than content quality."
    ],
    default="Multiple signals may not indicate a real intervention opportunity."
)

weak_cols = [
    "rank",
    "content_id",
    "score",
    "reason_code",
    "action",
    "why_it_may_be_weak"
]

weak_review = weak_picks[weak_cols]

print(f"Review threshold (90th percentile): {review_threshold:.2f}")
print("\nWeak-pick review:")
display(weak_review)

Review threshold (90th percentile): 64.92

Weak-pick review:


,rank,content_id,score,reason_code,action,why_it_may_be_weak
3006,3007,content_60d81e7dd1eb,64.897671,stale_content,monitor,Staleness alone may not mean the page needs a ...
3007,3008,content_0ae004d4a60f,64.892192,stale_content,monitor,Staleness alone may not mean the page needs a ...
3010,3011,content_2ace25e63c98,64.887260,stale_content,monitor,Staleness alone may not mean the page needs a ...
3008,3009,content_8517b9b89fbf,64.887260,stale_content,monitor,Staleness alone may not mean the page needs a ...
3009,3010,content_f0df99ec2996,64.887260,stale_content,monitor,Staleness alone may not mean the page needs a ...
3012,3013,content_e880205ae34e,64.887260,stale_content,monitor,Staleness alone may not mean the page needs a ...
3011,3012,content_21c803563f3e,64.887260,stale_content,monitor,Staleness alone may not mean the page needs a ...
3021,3022,content_09e7feae3fab,64.887260,stale_content,monitor,Staleness alone may not mean the page needs a ...
3018,3019,content_c66523d94c76,64.887260,stale_content,monitor,Staleness alone may not mean the page needs a ...
3013,3014,content_ab19c4527afd,64.887260,stale_content,monitor,Staleness alone may not mean the page needs a ...


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.